# 05b — Emotion timeseries of a text (vectors as a decoder)

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

nb04 built one **direction per emotion**. Here we *use* them: run an arbitrary text through the **same target model**, read its residual stream token-by-token, and **project every emotion vector** onto each token. The result is a **(token × emotion)** timeseries — a multivariate read-out of how each emotion rises and falls as the text unfolds.

$$\text{TS}[t, e] = \big(\,f(t) - \text{baseline}\,\big)\cdot \hat{v}_e$$

where `f(t)` is the residual at token *t* (at the projection layer) and `v̂_e` the unit emotion vector. We then draw:

1. a **line plot** of the top-K emotions peaking/diminishing over the text;
2. the **emotion × token** heatmap (`imshow`, z-scored per emotion);
3. the **token × token** correlation (which positions share an emotion profile);
4. the **emotion × emotion** correlation (which emotions co-vary along the text), ordered by cluster.

> Uses the **same model and conventions as nb04** — same projection layer, same `skip_first` sink handling, same baseline — read straight from `emotion_vectors.npz`, so the timeseries lives in the space the vectors were defined in.

## 0 — Setup

Like nb04, this notebook runs the target model, so it needs `torch` + `transformers` (and `bitsandbytes`/`accelerate` for big 4-bit targets). Portable bootstrap: Drive on Colab, `EMOVEC_WORK_DIR` on HPC/local.

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30",
                    "bitsandbytes>=0.43", "scikit-learn"], check=False)


In [ ]:
import json, os, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def _env(name, default):
    v = os.environ.get(name)
    return v if v not in (None, "") else default

def _env_int(name, default):
    v = os.environ.get(name)
    return int(v) if v not in (None, "") else default

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

if "HF_TOKEN" not in os.environ and IN_COLAB:
    try:
        from google.colab import userdata
        _t = userdata.get("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
    except Exception:
        pass

MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()

DATA_DIR = WORK_DIR / "data" / "processed"
FIG_DIR = WORK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"; VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    DEVICE, VRAM_GB = "cpu", 0.0

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"WORK_DIR={WORK_DIR}   device={DEVICE}")


## 1 — Load the emotion vectors (defines the model + layer)

Auto-discovers nb04's `features/{spec_hash}/{target_model}/` with the **most segments** (a real-model run, e.g. Qwen-7B, always outranks a gpt2 demo); candidates are listed so the pick is visible. Override with `EMOVEC_FEATURES_DIR` (exact dir) or `EMOVEC_TARGET_MODEL` (substring filter). The **target model, projection layer, baseline and `skip_first`** all come from here so 05b matches nb04 exactly.

In [ ]:
FEATURES_DIR = _env("EMOVEC_FEATURES_DIR", "")
PREFER_MODEL = _env("EMOVEC_TARGET_MODEL", "")   # substring filter, e.g. "Qwen"
if FEATURES_DIR:
    feat_dir = Path(FEATURES_DIR).expanduser()
else:
    manifests = sorted((DATA_DIR / "features").rglob("features_manifest.json"))
    assert manifests, f"no features under {DATA_DIR/'features'} — run nb04 first."
    def _info(p):
        try:
            m = json.loads(p.read_text())
            return m.get("target_model", p.parent.name), m.get("n_segments", 0)
        except Exception:
            return p.parent.name, 0
    for p in manifests:
        tm, n = _info(p)
        print(f"candidate: {tm:<36s} n_segments={n:<6d} {p.parent}")
    pool = [p for p in manifests if PREFER_MODEL in _info(p)[0]] if PREFER_MODEL else manifests
    assert pool, f"no features match EMOVEC_TARGET_MODEL={PREFER_MODEL!r}"
    feat_dir = max(pool, key=lambda p: _info(p)[1]).parent

vec = np.load(feat_dir / "emotion_vectors.npz", allow_pickle=True)
V            = vec["vectors"]                  # (E, n_layers, d)
emotions     = list(vec["emotions"])
cluster_lab  = vec["cluster_labels"]
baseline     = vec["baseline_mean"]            # (n_layers, d)
CLUSTER_L    = int(vec["cluster_layer"])
TARGET_MODEL = str(vec["target_model"])
SKIP_FIRST   = bool(vec["skip_first"]) if "skip_first" in vec.files else True
E, n_layers, d_model = V.shape

PROJ_LAYER = (int(_env("EMOVEC_PROJ_LAYER", "")) if _env("EMOVEC_PROJ_LAYER", "")
              else CLUSTER_L)
print(f"features dir : {feat_dir}")
print(f"emotions={E}  layers={n_layers}  d={d_model}  target={TARGET_MODEL}")
print(f"projection layer={PROJ_LAYER}  skip_first={SKIP_FIRST}")


## 2 — The text

A deliberately long passage with a **shifting emotional arc** (calm → joy → fear → grief → hope) so the timeseries has something to show. Override with `EMOVEC_TEXT` (inline) or `EMOVEC_TEXT_FILE` (a path).

In [ ]:
DEFAULT_TEXT = """
The morning began in stillness, a grey calm settling over the kitchen like a held breath. She poured her tea slowly and watched the steam curl toward the window. Nothing was urgent. Nothing was wrong. For a long, quiet moment the world asked nothing of her, and she let herself simply be, content in the small warmth of an ordinary day.

Then the letter came. She tore it open without thinking, and the words leapt up at her: she had won. A laugh broke out of her before she understood it, bright and helpless and loud. She spun once in the middle of the room, pressed the paper to her chest, read it again to be sure. Joy poured through her like sunlight through a cracked door, and she wanted to call everyone she had ever loved.

But the second page changed the air. A line she had not seen, a date, a name. And then a sound — footsteps in the hall where there should have been none. Her breath caught. The shadows along the wall seemed to lean toward her, and her heart slammed against her ribs, fast and frightened. She did not move. She could not. Every nerve strained toward the door, waiting for the handle to turn.

When at last she opened it, there was only an old photograph on the floor, and the terrible news folded behind it: her friend was gone. The room blurred and tilted. She sank down against the cold frame of the door, hollowed out, the grief rising in her chest until it ached to breathe. She wept without sound, the way people do when the loss is too large for noise.

And yet, in the silence afterward, something small and stubborn began to stir. She wiped her face. She stood. The grey light at the window had softened into something almost gold, and she found, against everything, a thin thread of hope to hold. Tomorrow she would begin again.
"""

TEXT = _env("EMOVEC_TEXT", "")
if not TEXT:
    tf = _env("EMOVEC_TEXT_FILE", "")
    TEXT = Path(tf).expanduser().read_text() if tf else DEFAULT_TEXT
TEXT = TEXT.strip()
MAX_TOKENS = _env_int("EMOVEC_MAX_TOKENS", 512)
print(f"text: {len(TEXT)} chars, ~{len(TEXT.split())} words (truncated to {MAX_TOKENS} tokens)")
print(TEXT[:300], "...")


## 3 — Load the target model and read its residual stream

Same loader as nb04 (HF `output_hidden_states`, with 4-bit/`device_map` for large targets). We pull the residual at `PROJ_LAYER` for every token. **Sink handling is measured, not assumed**: on a cold raw text the massive-activation transient can span *several* leading tokens (a fixed 1-token drop left ~5000-magnitude spikes dominating every figure), so we drop the leading run of tokens whose residual norm exceeds 3× the sequence median, and warn if any high-norm tokens remain mid-text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

PRECISION  = _env("EMOVEC_PRECISION", "auto")
DEVICE_MAP = _env("EMOVEC_DEVICE_MAP", "auto")

def _resolve_precision(pref):
    if pref != "auto":
        return pref
    if DEVICE == "cpu":
        return "fp32"
    return "4bit" if (VRAM_GB and VRAM_GB < 24) else "bf16"

prec = _resolve_precision(PRECISION)
hf_token = os.environ.get("HF_TOKEN")
tok = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True, token=hf_token)
kw = dict(trust_remote_code=True, token=hf_token, output_hidden_states=True)
if DEVICE == "cpu":
    kw.update(torch_dtype=torch.float32)
elif prec in ("4bit", "8bit") and BitsAndBytesConfig is not None:
    kw.update(quantization_config=BitsAndBytesConfig(
        load_in_4bit=(prec == "4bit"), load_in_8bit=(prec == "8bit"),
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True), device_map=DEVICE_MAP)
else:
    dtype = (torch.bfloat16 if prec == "bf16" and torch.cuda.is_bf16_supported()
             else torch.float16)
    kw.update(torch_dtype=dtype, device_map=DEVICE_MAP)
model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, **kw)
model.eval()
print(f"loaded {TARGET_MODEL} (precision={prec})")


In [ ]:
@torch.no_grad()
def resid_at_layer(text, layer):
    'Return (seq, d) residual at `layer` + token strings, sink-dropped if skip_first.'
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)
    dev = model.get_input_embeddings().weight.device
    enc = {k: v.to(dev) for k, v in enc.items()}
    hs = model(**enc).hidden_states                 # tuple len n_layers+1
    R = hs[layer + 1][0].cpu().float().numpy()      # (seq, d)   (+1 skips embeddings)
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())
    # Massive-activation "sink" tokens sit at the start of a cold sequence — and
    # on raw text the transient can span SEVERAL leading tokens, not just one
    # (a fixed 1-token drop left ~5000-magnitude spikes in the figures). So:
    # measure per-token norms and drop the leading run that exceeds 3x the
    # sequence median (at least 1 when nb04 used skip_first, to match spaces).
    nrm = np.linalg.norm(R, axis=1)
    med = float(np.median(nrm))
    n_lead = 0
    while n_lead < R.shape[0] - 1 and nrm[n_lead] > 3 * med:
        n_lead += 1
    if SKIP_FIRST:
        n_lead = max(n_lead, 1)
    R, toks, nrm = R[n_lead:], toks[n_lead:], nrm[n_lead:]
    mid = np.where(nrm > 3 * med)[0]                # any sinks left mid-text?
    return R, toks, n_lead, med, mid

R, toks, n_dropped, med_norm, mid_sinks = resid_at_layer(TEXT, PROJ_LAYER)
toks = [str(t).replace("Ġ", " ").replace("Ċ", "/n").strip() or t for t in toks]
print(f"residual: {R.shape}  ({len(toks)} tokens at layer {PROJ_LAYER})")
print(f"leading sink run: dropped {n_dropped} token(s) with norm > 3x median ({med_norm:.1f})")
if len(mid_sinks):
    print(f"WARN: {len(mid_sinks)} high-norm token(s) remain mid-text at positions "
          f"{mid_sinks[:10].tolist()} — inspect before trusting those stretches.")


## 4 — Project the emotion vectors → (token × emotion) timeseries

In [ ]:
def _unit(v, axis=-1, eps=1e-12):
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)

vhat = _unit(V[:, PROJ_LAYER, :])                   # (E, d) unit emotion dirs
base = baseline[PROJ_LAYER]                          # (d,)
TS = (R.astype(np.float64) - base) @ vhat.T          # (seq, E)  emotion timeseries
# z-score per emotion across tokens for display / correlation-friendly scaling.
TSz = (TS - TS.mean(0)) / (TS.std(0) + 1e-12)
print(f"emotion timeseries TS: {TS.shape}  (tokens x emotions)")


## 5 — Figure · emotion timeseries (top-K emotions)

The `EMOVEC_TOP_K` emotions with the strongest peaks, lightly smoothed (`EMOVEC_SMOOTH`) so the rise/fall is legible. Watch the arc move across the emotion families as the passage shifts mood.

In [ ]:
TOP_K  = _env_int("EMOVEC_TOP_K", 10)
SMOOTH = _env_int("EMOVEC_SMOOTH", 3)

def smooth(y, w):
    if w <= 1:
        return y
    k = np.ones(w) / w
    return np.convolve(y, k, mode="same")

# Rank by *raw* loading variation: v̂ are unit vectors, so raw projections are
# comparable across emotions. (Ranking by z-scored peak would surface flat/noisy
# emotions, since z-scoring inflates every emotion to unit variance.)
strength = TS.std(0)
top = np.argsort(strength)[::-1][:min(TOP_K, E)]
TSc = TS - TS.mean(0)                                # centre per emotion, keep amplitude
seq = TS.shape[0]
t = np.arange(seq)

fig, ax = plt.subplots(figsize=(max(8, 0.06 * seq + 3), 4.2))
cmap = plt.cm.tab20
for n, e in enumerate(top):
    ax.plot(t, smooth(TSc[:, e], SMOOTH), lw=1.6, color=cmap(n % 20),
            label=emotions[e])
ax.axhline(0, color="k", lw=0.5, alpha=0.4)
ax.set_xlabel("token position"); ax.set_ylabel("emotion loading (centred)")
ax.set_title(f"Emotion timeseries — top {len(top)} of {E} ({TARGET_MODEL}, layer {PROJ_LAYER})")
ax.legend(frameon=False, fontsize=7, ncol=2, loc="upper right")
# sparse token labels along the bottom for orientation.
step = max(1, seq // 25)
ax.set_xticks(range(0, seq, step))
ax.set_xticklabels([toks[i] for i in range(0, seq, step)], rotation=90, fontsize=6)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05b_fig1_timeseries.png", bbox_inches="tight")
plt.show()
print("strongest emotions:", ", ".join(emotions[e] for e in top))


## 6 — Figure · emotion × token heatmap

All emotions at once: `imshow` of the z-scored timeseries (`emotion` on y, `token` on x). Bright horizontal bands that switch as the passage progresses are the mood shifts.

In [ ]:
order_emo = np.argsort(cluster_lab, kind="stable")    # group emotions by cluster
M = TSz[:, order_emo].T                                 # (E, seq)
vmax = float(np.percentile(np.abs(M), 99)) or 1.0

fig, ax = plt.subplots(figsize=(max(8, 0.05 * seq + 3), max(4, 0.045 * E + 2)))
im = ax.imshow(M, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
               interpolation="nearest")
ax.set_xlabel("token position"); ax.set_ylabel("emotion (ordered by cluster)")
ax.set_title(f"Emotion x token timeseries (z) — {TARGET_MODEL}, layer {PROJ_LAYER}")
if E <= 60:
    ax.set_yticks(range(E)); ax.set_yticklabels([emotions[i] for i in order_emo], fontsize=5)
step = max(1, seq // 30)
ax.set_xticks(range(0, seq, step))
ax.set_xticklabels([toks[i] for i in range(0, seq, step)], rotation=90, fontsize=6)
fig.colorbar(im, ax=ax, shrink=0.8, label="loading (z)")
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05b_fig2_heatmap.png", bbox_inches="tight")
plt.show()


## 7 — Figure · correlation matrices

- **token × token** (`np.corrcoef(TS)`): which positions share an emotion *profile* — block structure marks stretches of the text that feel alike.
- **emotion × emotion** (`np.corrcoef(TS.T)`, ordered by cluster): which emotions rise and fall together across the text — the diagonal blocks should echo the nb04 clusters.

In [ ]:
tok_corr = np.corrcoef(TS)                      # (seq, seq)
emo_corr = np.corrcoef(TS.T)[np.ix_(order_emo, order_emo)]   # (E, E) cluster-ordered

fig, (axT, axE) = plt.subplots(1, 2, figsize=(12, 5.6))
imT = axT.imshow(tok_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
axT.set_title("token x token correlation (across emotions)")
axT.set_xlabel("token"); axT.set_ylabel("token")
step = max(1, seq // 25)
axT.set_xticks(range(0, seq, step)); axT.set_xticklabels([toks[i] for i in range(0, seq, step)],
                                                         rotation=90, fontsize=5)
axT.set_yticks(range(0, seq, step)); axT.set_yticklabels([toks[i] for i in range(0, seq, step)],
                                                         fontsize=5)
fig.colorbar(imT, ax=axT, shrink=0.7, label="r")

imE = axE.imshow(emo_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
axE.set_title("emotion x emotion correlation (across tokens, cluster-ordered)")
# cluster boundaries
bnd = np.where(np.diff(cluster_lab[order_emo]) != 0)[0] + 0.5
for b in bnd:
    axE.axhline(b, color="k", lw=0.3); axE.axvline(b, color="k", lw=0.3)
if E <= 60:
    axE.set_xticks(range(E)); axE.set_xticklabels([emotions[i] for i in order_emo], rotation=90, fontsize=5)
    axE.set_yticks(range(E)); axE.set_yticklabels([emotions[i] for i in order_emo], fontsize=5)
fig.colorbar(imE, ax=axE, shrink=0.7, label="r")
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05b_fig3_correlations.png", bbox_inches="tight")
plt.show()


## 8 — Notes + next

- This treats the emotion vectors as a **linear decoder** of the residual stream — fast, but uncalibrated: magnitudes across emotions aren't directly comparable, hence the per-emotion z-scoring for display. nb06 (held-out scoring) is where calibration/validation belongs.
- **Demo caveat:** with `gpt2` + a partial nb04 run the arcs are noisy; the method is what's being shown. Use the same real target you built the vectors with (`EMOVEC_TARGET_MODEL` in nb04) and a fuller emotion set for clean arcs.
- Layer matters: `EMOVEC_PROJ_LAYER` defaults to nb04's cluster layer. Mid layers usually carry the cleanest emotion geometry; sweep it if the arcs look flat.
- Same conventions as nb04 (projection layer, baseline, `skip_first`) so the timeseries is in the vectors' own space.